# Phase 3: Model Evaluation (mAP)

This notebook rigorously evaluates the `best.pt` checkpoint of the Hybrid YOLO-Swin Weapon Detector on your local validation dataset using `torchmetrics` (the industry standard for PyTorch). It calculates mAP50, mAP50-95, and mAR.

## 1. Environment Setup

In [ ]:
!pip install torchmetrics tqdm

## 2. Load the Model
Load the `best.pt` checkpoint from the local `models/weights/` directory.

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import sys

# Ensure the src/ and models/ directories are in the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from models.hybrid_model import HybridWeaponDetector

WEIGHTS_PATH = "../models/weights/best.pt"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model on {device}...")
model = HybridWeaponDetector.load(WEIGHTS_PATH, device=device)
model.eval()
print("Model loaded successfully!")

## 3. Dataset Loader
We define a custom PyTorch Dataset to parse the YOLO format `.txt` label files and yield `(image, annotations)` pairs safely, without relying on internal library functions.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class YOLOValidationDataset(Dataset):
    def __init__(self, images_dir):
        self.images_dir = Path(images_dir)
        self.labels_dir = self.images_dir.parent / "labels"
        self.image_paths = sorted(list(self.images_dir.glob("*.jpg")) + list(self.images_dir.glob("*.png")))
        
        print(f"Found {len(self.image_paths)} images in {self.images_dir}")
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(str(img_path))
        
        label_path = self.labels_dir / f"{img_path.stem}.txt"
        boxes = []
        labels = []
        
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        cx, cy, w, h = map(float, parts[1:])
                        # Convert [cx, cy, w, h] normalized to absolute pixel coordinates [x1, y1, x2, y2]
                        img_h, img_w = img.shape[:2]
                        x1 = (cx - w / 2) * img_w
                        y1 = (cy - h / 2) * img_h
                        x2 = (cx + w / 2) * img_w
                        y2 = (cy + h / 2) * img_h
                        boxes.append([x1, y1, x2, y2])
                        labels.append(class_id)
                        
        if len(boxes) == 0:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            
        target = dict(boxes=boxes, labels=labels)
        return img, target

# Point to local processed dataset
VAL_IMAGES_DIR = "../data/processed/yolo_dataset/val/images"
val_dataset = YOLOValidationDataset(VAL_IMAGES_DIR)

## 4. Quantitative Evaluation (mAP)
Compute mAP over the validation set using `torchmetrics.detection.MeanAveragePrecision`.

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision(box_format='xyxy', class_metrics=True)

print("Starting evaluation over validation set...")
for i in tqdm(range(len(val_dataset))):
    img, target = val_dataset[i]
    
    # Format ground truth
    gt = [{
        "boxes": target["boxes"].to(device),
        "labels": target["labels"].to(device)
    }]
    
    # Run Inference
    # model.predict automatically returns decoded, thresholded detections in pixel coordinates [x1, y1, x2, y2]
    # if it doesn't, we will ensure they are in the correct format.
    with torch.no_grad():
        detections = model.predict(img, conf_threshold=0.25, iou_threshold=0.45)
        
    pred_boxes = []
    pred_scores = []
    pred_labels = []
    
    img_h, img_w = img.shape[:2]
    for det in detections:
        bbox = det["bbox"]
        # Handle normalized vs pixel coordinates fallback
        if all(x <= 1.0 for x in bbox) and img_w > 1:
            px1, py1, px2, py2 = [bbox[0]*img_w, bbox[1]*img_h, bbox[2]*img_w, bbox[3]*img_h]
        else:
            px1, py1, px2, py2 = bbox
            
        pred_boxes.append([px1, py1, px2, py2])
        pred_scores.append(det["confidence"])
        pred_labels.append(det["class_id"])
        
    if len(pred_boxes) == 0:
        pred_boxes = torch.empty((0, 4), dtype=torch.float32).to(device)
        pred_scores = torch.empty((0,), dtype=torch.float32).to(device)
        pred_labels = torch.empty((0,), dtype=torch.int64).to(device)
    else:
        pred_boxes = torch.tensor(pred_boxes, dtype=torch.float32).to(device)
        pred_scores = torch.tensor(pred_scores, dtype=torch.float32).to(device)
        pred_labels = torch.tensor(pred_labels, dtype=torch.int64).to(device)
        
    preds = [{
        "boxes": pred_boxes,
        "scores": pred_scores,
        "labels": pred_labels
    }]
    
    metric.update(preds, gt)

# Compute final metric
print("\nComputing metrics (this may take a few seconds)...")
results = metric.compute()

print("\n" + "="*40)
print("         EVALUATION RESULTS")
print("="*40)
print(f"mAP@50       : {results['map_50'].item():.4f}")
print(f"mAP@50:95    : {results['map'].item():.4f}")
print(f"mAR@max_dets : {results['mar_100'].item():.4f}")
print("-"*40)
class_names = ["Weapon", "Person", "Confuser"]
if 'map_per_class' in results:
    print("Per-class mAP@50:95:")
    for i, ap in enumerate(results['map_per_class']):
        # Depending on torchmetrics version, classes might not perfectly align if some are missing
        # But typically they do if all classes exist in the dataset.
        c_id = results['classes'][i].item()
        name = class_names[c_id] if c_id < len(class_names) else f"Class {c_id}"
        print(f"  {name}: {ap.item():.4f}")
print("="*40)

## 5. Qualitative Visualization
Draw bounding boxes on a few sample images to verify visually.

In [ ]:
def visualize_prediction(img_idx, conf_threshold=0.25):
    img, _ = val_dataset[img_idx]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    detections = model.predict(img, conf_threshold=conf_threshold)
    
    h, w = img.shape[:2]
    for det in detections:
        bbox = det['bbox']
        cls_name = det['class_name']
        conf = det['confidence']
        
        if all(x <= 1.0 for x in bbox) and w > 1:
            x1, y1, x2, y2 = [int(bbox[0]*w), int(bbox[1]*h), int(bbox[2]*w), int(bbox[3]*h)]
        else:
            x1, y1, x2, y2 = [int(x) for x in bbox]
            
        color = (255, 0, 0) if det.get('is_weapon') else (0, 255, 0)
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        label = f"{cls_name} {conf:.2f}"
        cv2.putText(img_rgb, label, (x1, max(y1 - 10, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"Predictions: {len(detections)} objects")
    plt.show()

# Visualize the first 3 images
for i in range(min(3, len(val_dataset))):
    visualize_prediction(i, conf_threshold=0.25)